# Experimentos de modelagem preditiva - Influenza A (municipio de Sao Paulo)

**RF004 - modelo preditivo.** Este notebook documenta o experimento de modelagem seguindo o template do Manual Interno de Desenvolvimento do TCC: variavel-alvo, periodos de treino/teste, protocolo de validacao, modelos comparados, metricas, testes de significancia e limitacoes.

> **Nota de honestidade metodologica.** As metricas apresentadas sao as efetivamente obtidas pelo script `scripts/train_model.py`. A serie e curta e esparsa (casos-proxy derivados de `PCR_FLUASU`, hiato de 2020-2021). O Random Forest **nao** e forcado a superar o baseline; o resultado real e reportado tal como observado, conforme exige o Manual Interno (proibicao de apresentar o modelo como comprovadamente superior sem comparacao real).

Para regenerar todos os artefatos (modelo `.joblib`, grafico, tabela de metricas e registro no banco):

```bash
uv --directory backend run python ../scripts/train_model.py
```

## 1. Definicao do experimento

- **Variavel-alvo:** contagem semanal de casos-proxy de Influenza A no municipio de Sao Paulo (agregacao por semana epidemiologica). O proxy segue a regra do ETL (FEAT-001): registros com `PCR_FLUASU` preenchido contam como caso.
- **Granularidade:** municipal (nao ha granularidade por unidade/bairro na fonte consolidada).
- **Horizonte de previsao:** 6 semanas.
- **Base de treino:** semanas com `USAR_NO_TREINO = True` (as 8 semanas epidemiologicas mais recentes ficam reservadas por atraso de notificacao e nao entram no treino nem na avaliacao).
- **Protocolo de validacao:** cronologico *walk-forward* com origem expansiva (nunca aleatorio). A cada passo o modelo treina com todo o historico ate a origem e preve o proximo bloco de 6 semanas; a origem avanca 6 semanas e o processo se repete.
- **Modelos comparados:** baseline ingenuo (media sazonal por semana), Random Forest (principal), SARIMA e Prophet, todos sob o mesmo protocolo.

In [ ]:
# Ambiente: executar a partir de notebooks/ com o venv do backend.
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "backend"))
sys.path.insert(0, str(RAIZ / "scripts"))

import pandas as pd

serie = pd.read_csv(RAIZ / "data" / "processed" / "serie_temporal_semanal.csv")
treino = serie[serie["USAR_NO_TREINO"]].copy()
print("Semanas totais:", len(serie))
print("Semanas de treino:", len(treino))
print("Periodo de treino:", (treino["ANO"].min(), treino["ANO"].max()))
print("Semanas com zero casos (treino):", int((treino["CASOS_PROXY"] == 0).sum()))
treino.groupby("ANO")["CASOS_PROXY"].sum()

## 2. Engenharia de atributos temporais

O mesmo gerador de atributos e usado no treino e na previsao em producao (modulo `app.model.features`):

- **defasagens (lags):** 1, 2, 3 e 4 semanas;
- **medias moveis:** janelas de 3 e 5 semanas (apenas sobre o passado);
- **sazonalidade:** seno e cosseno do ciclo anual (52 semanas) e o mes aproximado da semana.

Todos os atributos de uma semana `t` usam somente informacao anterior a `t` (nao ha vazamento de informacao futura).

In [ ]:
from app.model import CONFIG_ATRIBUTOS_PADRAO, construir_matriz_treino

valores = treino["CASOS_PROXY"].astype(float).tolist()
semanas = treino["SEMANA_EPI"].astype(int).tolist()
matriz = construir_matriz_treino(valores, semanas, CONFIG_ATRIBUTOS_PADRAO)
print("Atributos:", matriz.nomes_atributos)
print("Observacoes supervisionadas:", len(matriz.X))
pd.DataFrame(matriz.X[:5], columns=matriz.nomes_atributos)

## 3. Avaliacao walk-forward e metricas

Reutilizamos a funcao do script de treino para avaliar todos os modelos sob o mesmo protocolo. (A celula abaixo pode levar alguns minutos por causa do Prophet e do SARIMA.)

In [ ]:
import train_model

avaliacoes = train_model.avaliar_modelos(valores, semanas)
testes = train_model.testar_significancia(avaliacoes)

linhas = []
for nome, dados in avaliacoes.items():
    m = dados["metricas"]
    linhas.append(
        {
            "modelo": nome,
            "MAE": round(m["mae"], 3),
            "RMSE": round(m["rmse"], 3),
            "sMAPE(%)": round(m["smape"], 2),
            "acerto_direcional": round(m["acerto_direcional"], 3),
        }
    )
pd.DataFrame(linhas)

## 4. Testes de significancia (Diebold-Mariano e Wilcoxon)

Verificam se a diferenca de acuracia entre o Random Forest e cada comparador e estatisticamente significativa. Diebold-Mariano com perda quadratica e correcao de amostra pequena (Harvey-Leybourne-Newbold); Wilcoxon como alternativa nao parametrica sobre os erros absolutos pareados.

In [ ]:
for nome, res in testes.items():
    print(f"RF vs {nome}:")
    print("  ", res["diebold_mariano"].detalhe)
    print("  ", res["wilcoxon"].detalhe)

## 5. Grafico real x previsto

O grafico completo e salvo em `docs/grafico_real_x_previsto.png` pelo script de treino.

In [ ]:
from IPython.display import Image

Image(filename=str(RAIZ / "docs" / "grafico_real_x_previsto.png"))

## 6. Resultado observado e limitacoes

Consulte `docs/metricas_modelos.md` para a tabela comparativa completa (incluindo p-valores) e `docs/documentacao_modelo.md` para a discussao detalhada das limitacoes.

**Sintese honesta do experimento.** Na serie municipal, curta e esparsa, o Random Forest **nao superou** o baseline sazonal-ingenuo em MAE/RMSE, e o teste de Diebold-Mariano **nao** indicou diferenca estatisticamente significativa entre eles. O SARIMA obteve o menor MAE, tambem sem diferenca significativa frente ao RF. Esses resultados sao reportados exatamente como obtidos.

**Principais limitacoes** (detalhadas em `docs/documentacao_modelo.md`):
- subnotificacao e atraso de notificacao (as 8 semanas mais recentes sao reservadas);
- serie curta e esparsa, com muitas semanas de zero casos e hiato de 2020-2021;
- alvo baseado em **proxy** (`PCR_FLUASU`), pois os campos de confirmacao laboratorial do TC2 nao existem na fonte;
- horizonte de 6 semanas com previsao recursiva acumula erro ao longo do horizonte.

O artefato serializado (`models/modelo_rf_v1.joblib`) e mesmo assim entregue e integrado a API (`/api/previsoes`), com as metricas reais registradas no banco e servidas por `/api/metricas`, cumprindo o requisito RF004 com transparencia sobre o desempenho.